# Introduction

# Installations, imports, utils

In [ ]:
!pip install transformers==4.33.0 accelerate==0.22.0 einops==0.6.1 langchain==0.0.300 xformers==0.0.21 \
bitsandbytes==0.41.1 sentence_transformers==2.2.2 chromadb==0.4.12

In [ ]:
from torch import cuda, bfloat16
import torch
import transformers
from transformers import AutoTokenizer
from time import time
from langchain.llms import HuggingFacePipeline
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma


# Initialize model, tokenizer, query pipeline

In [ ]:
model_id = '/kaggle/input/llama-2/pytorch/7b-chat-hf/1'
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

In [ ]:
model = transformers.AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, config=model_config, quantization_config=bnb_config, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
query_pipeline = transformers.pipeline('text-generation', model=model, tokenizer=tokenizer, torch_dtype=torch.float16, device_map='auto')

# Retrieval Augmented Generation

In [ ]:
llm = HuggingFacePipeline(pipeline=query_pipeline)
loader = TextLoader('/kaggle/input/president-bidens-state-of-the-union-2023/biden-sotu-2023-planned-official.txt', encoding='utf8')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
all_splits = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', model_kwargs={ 'device': 'cuda'})
vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory='chroma_db')
retriever = vectordb.as_retriever()
qa = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever, verbose=True)